# X-DETR — Colab Free (T4) training
Train the custom X-ray detector on OPIXray with per-epoch Google-Drive checkpointing so a
disconnect/restart resumes automatically. Runtime > Change runtime type > **T4 GPU**.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. Mount Drive
The project (already `git clone`d) and the dataset both live under your Drive project
folder so everything survives disconnects. Adjust `PROJ` below if your clone is elsewhere —
check the Colab file browser (left sidebar) under `drive/MyDrive/` to confirm the path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJ = '/content/drive/MyDrive/x-ray'   # <-- your existing clone; adjust if different
DATA = f'{PROJ}/data/OPIXray'           # matches configs/xdetr_opixray.yaml's default dataset.root
OUT  = f'{PROJ}/runs/opixray_xdetr'
os.chdir(PROJ); print('cwd:', os.getcwd())

In [ ]:
!git pull
!pip -q install scipy pyyaml gradio && echo done  # torch/torchvision preinstalled on Colab

## 2. Wiring sanity (no data needed) — should print PASS
This validates the model/matcher/loss end-to-end before you touch the dataset.

In [ ]:
!python -m scripts.overfit --config configs/xdetr_opixray.yaml --synthetic --iters 60

## 3. Get OPIXray into `$DATA`
OPIXray requires a signed academic agreement (see `scripts/download_opixray.md`) — it can't
be auto-downloaded from a generic script. Once you have YOUR OWN authorized copy (a direct
Drive file/zip, a local download, etc.), get it into `$DATA` using whichever applies:

* **You have a direct Google Drive share link/file ID** (e.g. from the authors): use
  `gdown` in the cell below — fill in your own `FILE_ID`.
* **Already sitting somewhere else in your Drive**: `!cp -r /path/to/OPIXray $DATA` or use
  the Colab file browser to move/rename it.
* **Zip on your local machine**: use a `google.colab.files.upload()` cell, then unzip into
  `$DATA`.

Expected layout after this step: `$DATA/train/train_image/*.jpg` +
`$DATA/train/train_annotation/*.txt` (and similarly under `test/`) — see
`scripts/download_opixray.md` for the tolerated variants.

In [ ]:
# Example using gdown with your own authorized Drive file ID (fill in FILE_ID):
FILE_ID = 'PUT_YOUR_OWN_FILE_ID_HERE'
!pip -q install gdown
!mkdir -p /content/_opixray_staging
!gdown --id $FILE_ID -O /content/_opixray_staging/opixray.zip
!unzip -q -o /content/_opixray_staging/opixray.zip -d /content/_opixray_staging/extracted
# Inspect BEFORE moving — the zip's internal folder structure may not match $DATA directly.
!find /content/_opixray_staging/extracted -maxdepth 3 | head -50

In [ ]:
# After confirming the structure above, move/rename the extracted folder to $DATA, e.g.:
# !mkdir -p $(dirname $DATA)
# !mv /content/_opixray_staging/extracted/<ACTUAL_TOP_FOLDER> $DATA
!ls -la $DATA

In [ ]:
!python -m scripts.sanity_data --config configs/xdetr_opixray.yaml --n 6 --out assets/sanity.png
from IPython.display import Image as IPyImage; IPyImage('assets/sanity.png')

## 4. Overfit 10 real images (boxes should visually snap) — correctness on real data

In [ ]:
!python -m scripts.overfit --config configs/xdetr_opixray.yaml --n 10 --iters 300

## 5. Train (T4). Checkpoints go to Drive every epoch
Re-running this cell after a disconnect **resumes automatically** from `last.pth`.

In [ ]:
!python -m engine.train --config configs/xdetr_opixray.yaml \
    --set training.output_dir=$OUT training.epochs=50 \
          model.dec_layers=6 model.num_queries=300

## 6. Evaluate (per-class AP, occlusion OL1/2/3, ECE)

In [ ]:
!python -m engine.evaluate --config configs/xdetr_opixray.yaml \
    --weights $OUT/last.pth --set training.output_dir=$OUT

## 7. Generate the visualization galleries

In [ ]:
!python -m scripts.gallery_batch --config configs/xdetr_opixray.yaml \
    --weights $OUT/last.pth --n 12 --out assets/galleries --score 0.3
from IPython.display import Image as IPyImage, display
import glob
for p in sorted(glob.glob('assets/galleries/*.png'))[:4]:
    display(IPyImage(p))

## 8. (Optional) Gradio demo with a public link
`--share` gives you a temporary public URL since Colab has no local browser.

In [ ]:
!python app/gradio_demo.py --config configs/xdetr_opixray.yaml --weights $OUT/last.pth --share